# 00 - Environment Check

**AI-Assisted Medical Image Analysis** - Chest X-ray classification & Brain MRI tumour segmentation


> **Academic prototype.** This notebook is part of a university final project.
> The models here are **not** medical devices, are **not** validated on clinical
> data, and must **never** be used to diagnose, screen or triage real patients.
> See `docs/ETHICS.md`.


Run this notebook **first**. It verifies that the environment can actually run
the project, before you spend time on data. Nothing here trains anything.

Checks: Python version -> required packages -> PyTorch & GPU -> project imports
-> folder structure -> reproducible seeding -> a tiny end-to-end forward pass.

In [ ]:
# Make `src/` importable no matter where Jupyter was launched from.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

print("Project root:", PROJECT_ROOT)

## 1. Python version

The project targets **Python 3.11**.

In [ ]:
import platform
import sys

print("Python  :", sys.version)
print("Platform:", platform.platform())
print("Executable:", sys.executable)

major, minor = sys.version_info[:2]
if (major, minor) != (3, 11):
    print(f"\n[warn] This project was written for Python 3.11; you are on {major}.{minor}.")
else:
    print("\n[ok] Python 3.11")

## 2. Required packages

Anything marked MISSING must be installed before continuing (see `README.md` -> Environment setup).

In [ ]:
import importlib.metadata as md_meta

REQUIRED = ["torch", "torchvision", "numpy", "pandas", "scipy", "scikit-learn",
            "matplotlib", "seaborn", "Pillow", "PyYAML", "tqdm", "ipykernel"]
OPTIONAL = ["opencv-python", "scikit-image", "ipywidgets", "jupyterlab"]

def report(names, kind):
    missing = []
    print(f"--- {kind} ---")
    for name in names:
        try:
            print(f"  {name:<16s} {md_meta.version(name)}")
        except md_meta.PackageNotFoundError:
            print(f"  {name:<16s} MISSING")
            missing.append(name)
    return missing

missing_required = report(REQUIRED, "required")
missing_optional = report(OPTIONAL, "optional")

print()
if missing_required:
    print("[FAIL] install these before continuing:")
    print("       pip install " + " ".join(missing_required))
else:
    print("[ok] all required packages present")
if missing_optional:
    print("[info] optional, only needed for a few convenience paths:", ", ".join(missing_optional))

## 3. PyTorch and GPU

A GPU is strongly recommended but not required - every notebook falls back to CPU (slower). `device: auto` in `configs/base.yaml` handles the choice.

In [ ]:
import torch

from src.common import describe_device, get_device

device = get_device("auto")
for key, value in describe_device(device).items():
    print(f"{key:<22s} {value}")

if not torch.cuda.is_available():
    print("\n[info] Running on CPU. Training will be slow; reduce train.epochs and "
          "data.image_size in the configs for a first pass.")

## 4. Project imports

Every reusable function lives in `src/` - the notebooks only orchestrate. If this cell fails, the working directory or `sys.path` is wrong.

In [ ]:
from src.common import load_config, seed_everything, ensure_dir
from src.classification import build_manifest, build_model, train_model, evaluate_classifier
from src.segmentation import build_pairs, build_segmentation_model, train_segmentation

print("[ok] src.common, src.classification and src.segmentation import cleanly")

## 5. Folder structure and configs

In [ ]:
EXPECTED = ["data/raw/xray", "data/raw/mri", "data/processed", "configs",
            "outputs/classification", "outputs/segmentation", "notebooks", "src", "docs"]

for rel in EXPECTED:
    path = PROJECT_ROOT / rel
    mark = "ok " if path.exists() else "MISSING"
    print(f"[{mark}] {rel}")

print()
for cfg_file in sorted((PROJECT_ROOT / "configs").glob("*.yaml")):
    print("config:", cfg_file.name)

In [ ]:
# Load each config to confirm the YAML is valid and inheritance resolves.
for name in ["xray_baseline.yaml", "xray_transfer.yaml", "mri_unet.yaml"]:
    cfg = load_config(name)
    print(f"{name:<22s} run_name={cfg.get('run_name'):<22s} "
          f"seed={cfg.get('seed')} image_size={cfg.get('data.image_size')}")

## 6. Is the data present?

The course dataset is added later. This cell just reports what is currently on disk - **empty is expected right now**.

In [ ]:
for name, rel in [("Chest X-ray", "data/raw/xray"), ("Brain MRI", "data/raw/mri")]:
    root = PROJECT_ROOT / rel
    entries = sorted(p.name for p in root.iterdir()) if root.exists() else []
    entries = [e for e in entries if e != ".gitkeep"]
    print(f"{name:<12s} {rel}")
    print(f"             {len(entries)} entries" + (f": {entries[:8]}" if entries else
          "  <- add your dataset here (see data/README.md)"))

## 7. Reproducible seeding

Seeding must give identical random draws on repeated calls. See `docs/REPRODUCIBILITY.md` for what seeding can and cannot guarantee.

In [ ]:
import numpy as np

seed_everything(42)
first = (np.random.rand(3).round(6).tolist(), torch.rand(3).round(decimals=6).tolist())

seed_everything(42)
second = (np.random.rand(3).round(6).tolist(), torch.rand(3).round(decimals=6).tolist())

print("run 1:", first)
print("run 2:", second)
print("\n[ok] seeding is reproducible" if first == second else "\n[FAIL] seeding is NOT reproducible")

## 8. End-to-end smoke test (synthetic tensors, no data needed)

Builds both model families and pushes random noise through them. This proves the architectures, shapes and the loss functions work **before** the dataset arrives.

*The numbers below are meaningless - the input is random noise, not medical images.*

In [ ]:
from src.classification.losses import build_loss
from src.classification.models import SimpleCNN
from src.segmentation.losses import DiceBCELoss
from src.segmentation.unet import UNet

seed_everything(42)

# --- classifier ---------------------------------------------------------
clf = SimpleCNN(num_labels=2).to(device).eval()
x = torch.randn(2, 3, 224, 224, device=device)
with torch.no_grad():
    logits = clf(x)
print(f"SimpleCNN     input {tuple(x.shape)} -> logits {tuple(logits.shape)}")

# --- U-Net --------------------------------------------------------------
unet = UNet(in_channels=3, out_channels=1, features=(16, 32, 64, 128)).to(device).eval()
xi = torch.randn(2, 3, 256, 256, device=device)
with torch.no_grad():
    seg_logits = unet(xi)
print(f"UNet(small)   input {tuple(xi.shape)} -> logits {tuple(seg_logits.shape)}")

# --- losses -------------------------------------------------------------
bce = torch.nn.BCEWithLogitsLoss()(logits, torch.randint(0, 2, logits.shape, device=device).float())
dice_bce = DiceBCELoss()(seg_logits, torch.randint(0, 2, seg_logits.shape, device=device).float())
print(f"\nBCEWithLogits on random tensors: {bce.item():.4f}")
print(f"DiceBCELoss   on random tensors: {dice_bce.item():.4f}")
print("\n[ok] forward passes and losses run - these values are noise, not results")

---

## Result

If every section above says `ok`, the environment is ready.

**Next:** add your datasets (see `data/README.md`), then run
`01_xray_eda_and_preparation.ipynb`.

### Screenshot for the report
- Section 3 output (PyTorch version + GPU) - documents the hardware your results were produced on.